# 01 · NumPy vs CuPy 벤치마크 기본

> **CuPy 2일 집중 코스 — Day 1 / 단원 1 (GPU 컴퓨팅과 CuPy 개론)**

GPU가 *언제* 빠른지를 **직접 측정**으로 배웁니다. 올바른 벤치마킹 방법을 익히고,
문제 크기에 따른 손익분기점과 전송 비용, 벡터화의 효과를 확인합니다.

### 왜 벤치마킹부터 정량화하는가
00에서 우리는 GPU 연산이 비동기라서 `time.perf_counter()`로 재면 착시가 생긴다는 사실과, 전송이
비쌀 수 있다는 사실을 눈으로 직접 확인했습니다. 하지만 00은 정성적(qualitative) 관찰에 머물렀습니다 —
"느려 보인다", "빨라 보인다" 정도였죠. 01은 그 관찰을 **숫자로** 만듭니다: 정확히 몇 배 빠른가?
몇 개의 원소부터 GPU가 유리한가? 전송을 포함하면 이득이 얼마나 줄어드는가? 이런 질문에 답하려면
먼저 '올바르게 재는 법'부터 확립해야 합니다 — 잘못된 자로 잰 숫자는 숫자여도 의미가 없습니다.

### 이 노트북의 흐름
벤치마킹 원칙(워밍업/동기화/반복평균) → 크기별 CPU/GPU 비교로 손익분기점 관찰 → 전송을 포함한
end-to-end 비용 → 벡터화의 중요성을 수치로 재확인. 이 순서는 실무에서 GPU 이식 여부를 판단할 때
실제로 거치는 절차와 같습니다: "이 연산, GPU로 옮길 가치가 있는가?"라는 질문에 감이 아니라
측정으로 답하는 법을 익히는 것이 이 노트북의 목표입니다.

## 학습 목표
- GPU 비동기 특성을 고려한 **올바른 벤치마킹**(워밍업·동기화·`benchmark`)을 수행한다.
- 문제 크기에 따라 GPU 이득이 달라지는 **손익분기점(break-even)** 을 찾는다.
- 전송 포함 **end-to-end** 비용과 연산만의 비용을 구분한다.
- **벡터화**가 (CPU·GPU 모두) 왜 필수인지 수치로 확인한다.

## 목차
1. [GPU 벤치마킹의 원칙](#1)
   - [1.1 일회성 오버헤드 & %gpu_timeit](#1)
2. [크기별 성능 비교 — 손익분기점](#2)
3. [전송 포함 end-to-end 비용](#3)
4. [벡터화의 힘](#4)
5. [연습문제](#5)
6. [체크포인트](#6)

> 참고: `course_utils.bench/compare/print_bench`는 모두 `cupyx.profiler.benchmark` 기반입니다. (단원 0에서 소개)

In [ ]:
import os, sys, time, math
import numpy as np
import cupy as cp
from course_utils import print_env, bench, gpu_ms, cpu_ms, print_bench, compare
print_env()

### 📦 `course_utils` 함수 — 이번 노트북에서 도입

실습 공통 유틸리티는 `course_utils.py`에 모아두고 노트북이 진행되며 **필요한 함수만 하나씩** 추가합니다. 01에서 도입하는 함수:

- **`print_bench(result)`** — `bench` 결과를 한 줄로 보기 좋게 출력 (CPU wall-clock / GPU 커널).
- **`compare(name, cpu_fn, gpu_fn, *, n_repeat: int = 10, n_warmup: int = 2)`** — CPU 함수와 GPU 함수를 같은 조건으로 측정해 wall-clock으로 공정 비교.

두 함수 모두 00에서 도입한 `bench`/`gpu_ms`/`cpu_ms` 위에 얇게 얹은 '보고용' 래퍼입니다 — 새로운
측정 원리는 없고, 반복해서 손으로 쓰던 출력 포맷·비교 로직을 함수로 뽑아낸 것뿐입니다. 아래 코드
셀에 실제 구현이 그대로 노출되어 있으니(00과 동일한 교육적 노출 방식), 주석을 함께 읽으면
`compare`가 **왜 GPU 쪽도 `gpu_ms`(커널시간)가 아니라 `cpu_ms`(wall-clock)로 비교하는지** — 공정한
end-to-end 비교를 위해서입니다 — 바로 확인할 수 있습니다.

In [ ]:
def print_bench(result) -> None:
    """bench() 결과를 한 줄로 보기 좋게 출력 (CPU wall-clock / GPU 커널)."""
    c = cpu_ms(result)  # CPU 관점 wall-clock 평균(ms): launch overhead·파이썬 오버헤드 포함
    g = gpu_ms(result)  # GPU 관점 커널 실행 평균(ms): 순수 커널 실행 시간만
    # result.name: bench() 호출 시 지정한 name(또는 함수명). >20/9.3f로 폭을 고정해
    # 여러 줄을 연달아 출력해도 표처럼 열이 맞춰져 보이게 함
    print(f"{result.name:>20} | CPU(wall) {c:9.3f} ms | GPU(kernel) {g:9.3f} ms")


In [ ]:
def compare(name, cpu_fn, gpu_fn, *, n_repeat: int = 10, n_warmup: int = 2):
    """CPU 함수와 GPU 함수를 같은 조건으로 측정해 wall-clock으로 공정 비교.

    GPU도 CPU(wall-clock) 시간으로 비교합니다(전송·동기화 포함, end-to-end 관점).
    Returns: (cpu_ms, gpu_ms, speedup)
    """
    # 두 함수(cpu_fn/gpu_fn)를 동일한 n_repeat/n_warmup 조건으로 각각 측정.
    # name에 "-cpu"/"-gpu" 접미사를 붙여 어느 쪽 결과인지 구분되게 함
    rc = bench(cpu_fn, n_repeat=n_repeat, n_warmup=n_warmup, name=f"{name}-cpu")
    rg = bench(gpu_fn, n_repeat=n_repeat, n_warmup=n_warmup, name=f"{name}-gpu")
    # 주의: GPU 쪽도 gpu_ms가 아닌 cpu_ms(wall-clock)를 사용한다.
    # 커널 시간(gpu_ms)만 비교하면 host<->device 전송·동기화 오버헤드가 감춰져 불공정한
    # 비교가 되므로, 사용자가 실제 체감하는 end-to-end(호출~반환) 시간으로 맞춰 비교한다.
    c = cpu_ms(rc)
    g = cpu_ms(rg)
    # GPU가 더 빠르면 speedup > 1. g가 0에 가까워 나눗셈이 위험한 경우 무한대로 대체
    speedup = c / g if g > 0 else float("inf")
    print(f"{name:>16} | CPU {c:9.3f} ms | GPU {g:9.3f} ms | {speedup:6.2f}x")
    return c, g, speedup


<a id="1"></a>
## 1. GPU 벤치마킹의 원칙

GPU 연산은 **비동기**입니다. 그래서 `time.perf_counter()`로 한 번 재면 부정확합니다. 올바른 측정의 3원칙:
1. **워밍업(warmup)**: 첫 호출엔 커널 컴파일·캐시 등 1회성 비용이 섞입니다. 몇 번 버린 뒤 측정.
2. **동기화(synchronize)**: 커널 완료를 기다린 뒤 시간을 읽습니다.
3. **반복 평균**: 여러 번 반복해 평균±표준편차로 봅니다.

`cupyx.profiler.benchmark`(= `course_utils.bench`)가 이 셋을 모두 처리합니다.
공정한 CPU↔GPU 비교를 위해 **wall-clock(`cpu_times`)** 을 기준으로 삼습니다(전송·동기화 포함, end-to-end 관점).

### 조금 더 구체적으로: 무엇을, 왜 재는가

세 원칙 각각이 왜 필요한지 00에서 확인한 내용을 다시 수치 관점에서 정리하면:

1. **워밍업이 필요한 이유**: CUDA 컨텍스트 최초 초기화는 수백 ms~수 초, shape/dtype별 커널 JIT
   컴파일은 수십~수백 ms가 걸릴 수 있습니다(1.1절에서 다시 다룸). 이 비용은 **딱 한 번만** 발생하므로,
   워밍업 없이 첫 실행을 측정치에 포함시키면 실제 반복 실행 시간을 심하게 과대평가하게 됩니다.
2. **동기화가 필요한 이유**: GPU는 커널을 스트림 큐에 '제출'만 받고 백그라운드에서 실행합니다
   (00 6절). `synchronize()` 없이 시간을 재면 커널이 끝나기도 전에 시계가 멈춰, 실제보다 훨씬
   빠르게(심하면 거의 0으로) 보입니다.
3. **반복 평균이 필요한 이유**: GPU 클럭 부스트, 다른 프로세스와의 자원 경합, OS 스케줄링 지터
   등으로 한 번의 측정은 노이즈가 큽니다. `benchmark`는 평균과 표준편차를 함께 보고해, "얼마나
   안정적으로 빠른가"까지 판단할 수 있게 합니다 — 표준편차가 평균에 비해 크다면 그 자체로 병목이
   워밍업 부족이나 시스템 경합일 수 있다는 신호입니다.

**wall-clock(`cpu_times`) vs 커널(`gpu_times`)의 차이**는 이 노트북 전체에서 반복해서 등장하는 구분입니다:

| 지표 | 포함하는 것 | 용도 |
|------|-------------|------|
| `gpu_times` (커널) | 커널이 GPU에서 실제로 실행된 시간만 | '이 연산 자체가 GPU에서 얼마나 걸리는가'를 순수하게 볼 때 |
| `cpu_times` (wall-clock) | 커널 launch overhead + 파이썬 오버헤드 + (필요 시) 전송·동기화 대기까지 | 사용자가 실제로 체감하는 end-to-end 시간, CPU 코드와 **공정하게 비교**할 때 |

`compare()`가 GPU 쪽도 `cpu_ms`(wall-clock)를 쓰는 이유가 바로 이것입니다 — 커널 시간만 비교하면
실제로는 존재하는 launch·전송 비용을 숨기고 GPU를 실제보다 유리하게 보이게 만듭니다. 2절·3절에서
이 구분이 어떻게 손익분기점을 좌우하는지 직접 확인합니다.

In [ ]:
# benchmark는 CPU(wall-clock)와 GPU(kernel) 시간을 함께 보고합니다.
a = cp.random.random(10_000_000, dtype=cp.float32)
r = bench(lambda: (a * 1.1 + 2.0).sum(), n_repeat=20, n_warmup=3, name='fused_sum')
print_bench(r)
print(r)   # benchmark 객체의 기본 출력(반복 통계 포함)

### 1.1 일회성 오버헤드 & `%gpu_timeit`

워밍업이 필요한 이유는 **일회성 오버헤드** 때문입니다(공식 문서 권고).
- 프로세스 첫 CUDA 호출 시 **컨텍스트 초기화**(수 초).
- 인자 shape/dtype별 **커널 JIT 컴파일**(이후 `~/.cupy/kernel_cache`에 캐시되어 재사용).

이 두 비용은 성격이 다릅니다. **컨텍스트 초기화**는 프로세스당 정확히 한 번(보통 1~3초, 드라이버·
GPU에 따라 다름)이라 워밍업 여부와 무관하게 노트북을 실행하는 순간 이미 한 번 치르고 지나간
비용입니다(00에서 `print_env()`를 처음 호출할 때 이미 발생했습니다). 반면 **커널 JIT 컴파일**은
shape/dtype **조합마다** 새로 발생합니다 — 예를 들어 `float32` 배열용으로 컴파일된 커널은 같은
연산이라도 `float64`나 다른 shape에는 재사용되지 않고 다시 컴파일됩니다. 디스크 캐시
(`~/.cupy/kernel_cache`, 환경변수 `CUPY_CACHE_DIR`로 위치 변경 가능)는 **같은 프로세스를 재시작해도**
컴파일을 건너뛸 수 있게 해주지만, 프로세스 내 첫 호출의 파이썬 오버헤드까지 없애주지는 않습니다.
그래서 `n_warmup`은 최소 2~3 이상을 권장합니다.

주피터/IPython에서는 `cupyx.profiler` 확장의 **`%gpu_timeit`** 매직으로 간단히 잴 수 있습니다(`benchmark`와 동일 옵션 `-n`,`-w`).

```python
%load_ext cupyx.profiler
%gpu_timeit -n 20 (cp.random.random(1_000_000, dtype=cp.float32) ** 2).sum()

%%gpu_timeit
x = cp.random.random((2000, 2000), dtype=cp.float32)
y = x @ x.T
```

`%gpu_timeit`은 IPython 매직인 `%timeit`의 GPU-aware 버전이라고 생각하면 됩니다:

| 매직/함수 | 동기화 처리 | 용도 |
|-----------|-------------|------|
| `%timeit` | 안 함(wall-clock만) | 순수 CPU 코드 |
| `%gpu_timeit` / `cupyx.profiler.benchmark` | CUDA 이벤트로 동기화 | GPU(비동기) 코드 |

`%timeit`으로 GPU 코드를 재면 00의 '비동기 함정'을 그대로 재현하게 되므로, GPU 코드는 항상
`%gpu_timeit` 또는 `bench`를 쓰세요.

<a id="2"></a>
## 2. 크기별 성능 비교 — 손익분기점

GPU는 **커널 런치 오버헤드**(수~수십 µs)와 **전송 비용**이 있어, 데이터가 작으면 CPU보다 느릴 수 있습니다.
GPU가 이기려면 **충분한 작업량(workload)** 이 필요합니다. 아래 도해의 직관:
연산이 `O(N)`이면 이득을 보려면 데이터가 커야 하고, 연산량이 `O(N)`보다 크면 더 적은 데이터로도 이득을 봅니다.

<img src="images/figures/new_latency_bandwidth.png" width="640">

### 조금 더 구체적으로: 손익분기점을 수식으로 보기

CPU와 GPU의 실행 시간을 문제 크기 N에 대한 선형 모델로 단순화하면 직관을 수식으로 잡을 수 있습니다.

```
T_cpu(N) ≈ a0 + a1·N        (a0: 파이썬 함수 호출 오버헤드, a1: 원소당 연산 비용)
T_gpu(N) ≈ b0 + b1·N        (b0: 커널 launch + 관련 고정 오버헤드, b1: 원소당 연산 비용)
```

일반적으로 `b0 > a0`(GPU 쪽 고정비가 더 큼: 런치 오버헤드가 수~수십 µs)이지만, 원소당 비용은
`b1 ≪ a1`(GPU의 코어 수·메모리 대역폭이 훨씬 크기 때문에 같은 연산이라도 원소 하나를 처리하는
실효 비용이 훨씬 작음)입니다. 두 직선이 만나는 지점, 즉 `a0 + a1·N* = b0 + b1·N*`을 풀면

```
N* = (b0 - a0) / (a1 - b1)
```

이 **손익분기점(break-even point)** 입니다. N < N\*이면 고정비 차이(`b0 - a0`)가 지배적이라 CPU가
유리하고, N > N\*이면 원소당 비용 차이(`a1 - b1`)가 누적되어 GPU가 유리해집니다. 아래 그림에서
'연산이 O(N)이면 데이터가 커야 한다'는 말은 정확히 `a1 - b1`의 차이가 **선형으로만** 벌어진다는
뜻이고, 뒤에 나올 행렬곱(`O(N^3)`, 실험 B)처럼 연산량이 데이터 크기보다 빠르게 늘어나는 경우는
그 차이가 훨씬 빨리 벌어져 **더 작은 N에서도** GPU가 유리해집니다 — 이것이 원소별 연산과 행렬곱의
손익분기점이 크게 다른 이유입니다.


In [ ]:
# 같은 연산 (a*1.1+b).sum() 을 크기별로 CPU vs GPU 비교 (wall-clock)
sizes = [10_000, 100_000, 1_000_000, 10_000_000, 50_000_000]
for n in sizes:
    a_np = np.random.rand(n).astype(np.float32)
    b_np = np.random.rand(n).astype(np.float32)
    a_cp, b_cp = cp.asarray(a_np), cp.asarray(b_np)
    cpu_fn = lambda a=a_np, b=b_np: (a * 1.1 + b).sum()
    gpu_fn = lambda a=a_cp, b=b_cp: (a * 1.1 + b).sum()
    compare(f'N={n:,}', cpu_fn, gpu_fn, n_repeat=10, n_warmup=2)

# => 작은 N에서는 speedup<1(GPU가 느림), N이 커질수록 speedup이 커집니다.

<a id="3"></a>
## 3. 전송 포함 end-to-end 비용

GPU 연산 자체는 빨라도, 매번 결과를 `asnumpy`로 host에 가져오면 **전송이 병목**이 됩니다.
'연산만' vs '연산+전송'을 비교해 전송 비용을 체감합니다. (메모리·전송 최적화는 단원 3에서 심화)

### 조금 더 구체적으로: 전송 비용의 두 성분

00 7절에서 이미 확인했듯 전송 시간은 대략

```
T_transfer(bytes) ≈ latency + bytes / bandwidth
```

형태를 따릅니다. `latency`는 전송 1회당 고정 비용(수 µs~수십 µs, 드라이버 호출·DMA 셋업 등),
`bandwidth`는 PCIe Gen4 x16 기준 이론상 편도 약 32 GB/s(실측은 보통 그 70~90% 수준)입니다. 이
노트북의 실험은 **스칼라 하나**(`.sum()`의 결과)만 전송하므로 `bytes`는 사실상 무시할 만큼
작고, 측정되는 오버헤드는 거의 전부 `latency` 성분입니다 — 즉 "가져올 데이터가 작으니 공짜"가
아니라, **전송 횟수 자체가 비용**이라는 뜻입니다. 이 사실은 실무에서 다음과 같은 함의를 가집니다:

> 루프 안에서 매 반복마다 `.item()`이나 `float(x)`, `print(x)`처럼 GPU 스칼라를 host 값으로
> 바꾸는 코드는(암묵적으로 동기화 + 전송을 유발) N번의 고정 지연을 그대로 누적시킵니다. 중간
> 결과는 GPU에 남겨두고 **최종 결과만, 딱 한 번** 가져오세요.

pinned(고정) 메모리로 전송 대역폭 자체를 끌어올리는 기법, 그리고 전송과 연산을 스트림으로
겹쳐(overlap) 지연을 감추는 기법은 `05_memory_profiling`·`06_streams_async`에서 각각 심화로
다룹니다.

In [ ]:
n = 10_000_000 # n을 변경해보세요.
a = cp.random.random(n, dtype=cp.float32)

r_compute = bench(lambda: (a * 2.0 + 1.0).sum(), n_repeat=20, name='compute_only')
r_e2e     = bench(lambda: cp.asnumpy((a * 2.0 + 1.0).sum()), n_repeat=20, name='compute+transfer')
print_bench(r_compute)
print_bench(r_e2e)
print('=> 결과 스칼라 하나를 가져오는 데도 전송 오버헤드가 붙습니다. 루프 안 반복 전송을 피하세요.')

<a id="4"></a>
## 4. 벡터화의 힘

GPU 가속의 **대전제**는 벡터화입니다. 파이썬 `for`로 원소를 하나씩 처리하면 NumPy의 최적화된 네이티브 연산을
활용하지 못하고, GPU에서는 특히 **수십~수백 배** 느려집니다. 항상 배열 연산으로 표현하세요.

<img src="images/figures/new_vectorize_vs_loop.png" width="600">

<img src="images/figures/new_avoid_serial_loops.png" width="600">

### 조금 더 구체적으로: 벡터화가 없으면 무슨 일이 일어나는가

파이썬 `for` 루프로 `out[i, j] = A[i, j] + A[i, j] ** 2`처럼 원소 하나씩 접근하면, 매 반복마다
(1) 인터프리터의 바이트코드 디스패치 비용(수십~수백 ns), (2) NumPy 배열 인덱싱이 매번 **파이썬
스칼라 객체를 새로 생성**하는 boxing 비용(대략 원소당 수백 ns~1 µs)이 누적됩니다. 100만 원소짜리
`1024×1024` 배열이면 이것만으로도 수백 ms~초 단위가 됩니다 — 벡터화된 `A + A**2` 한 줄이면
NumPy가 C 레벨 루프 하나로 처리해 같은 작업을 밀리초 단위로 끝내는 것과 극명하게 대비됩니다.

`@numba.njit`은 이 파이썬 인터프리터 오버헤드를 없애 **C 수준으로 컴파일된 루프**를 만들어주므로
`loop_fast`가 `loop`보다 수십~수백 배 빠릅니다 — 다만 여전히 CPU 루프이므로 완전히 벡터화된
NumPy/CuPy 연산보다는 대체로 느립니다.

만약 이 원소별 루프를 **GPU에서** 그대로(`for i in range(...): out_cp[i,j] = ...`) 실행한다면
문제는 더 심각해집니다. CuPy 배열의 인덱싱 대입 한 번은 사실상 커널 실행 한 번에 해당하므로,
원소 하나마다 **커널 런치 오버헤드(수~수십 µs, 2절에서 다룬 `b0`)** 를 반복해서 지불하게 됩니다.
100만 원소면 런치 오버헤드만 수십 초에 달할 수 있어, 벡터화된 GPU 연산(전체를 한 번의 커널로
처리)과 비교하면 수백~수천 배 느려지는 극단적인 경우도 생깁니다. "GPU에서 루프가 특히 위험하다"는
경고는 이런 이유 때문입니다 — CPU 루프는 '느리게라도' 진행되지만, GPU 원소별 루프는 매 원소가
launch 오버헤드 전액을 다시 치르는 구조라 자릿수 자체가 달라집니다.

> **실무 규칙**: GPU 코드에서 `for` 루프가 보인다면 십중팔구 안티패턴입니다. 배열 연산(`+`, `*`,
> `cp.sum`, 브로드캐스팅, 슬라이싱)으로 다시 표현할 수 있는지 먼저 확인하세요. 정말 원소별
> 커스텀 연산이 필요하다면 파이썬 루프 대신 `cupy.ElementwiseKernel`을 쓰는 법을 Day 2(단원 5~6)에서
> 배웁니다.

In [ ]:
import numba

# 같은 연산 A + A**2 : 파이썬 루프 vs 벡터화(CPU) vs 벡터화(GPU)
A = np.random.random((1024, 1024)).astype(np.float32) # n을 변경해보세요

def loop(A):
    out = np.empty_like(A)
    for i in range(A.shape[0]):
        for j in range(A.shape[1]):
            out[i, j] = A[i, j] + A[i, j] ** 2
    return out

@numba.njit(fastmath=True) # JIT 컴파일러에게 이 루프를 C수준으로 최적화하라고 명령!
def loop_fast(A):
    out = np.empty_like(A)
    for i in range(A.shape[0]):
        for j in range(A.shape[1]):
            out[i, j] = A[i, j] + A[i, j] ** 2
    return out

_ = loop_fast(A) # warm-up


t0 = time.perf_counter(); loop(A); t1 = time.perf_counter()
print(f'CPU 파이썬 루프 : {(t1 - t0) * 1e3:10.1f} ms')

t0 = time.perf_counter(); loop_fast(A); t1 = time.perf_counter()
print(f'NUMBA JIT 루프 : {(t1 - t0) * 1e3:10.1f} ms')

r_cpu = bench(lambda: A + A ** 2, n_repeat=10, name='cpu_vec')
print(f'CPU 벡터화      : {cpu_ms(r_cpu):10.3f} ms')

A_cp = cp.asarray(A)
r_gpu = bench(lambda: A_cp + A_cp ** 2, n_repeat=20, name='gpu_vec')
print(f'GPU 벡터화      : {cpu_ms(r_gpu):10.3f} ms')

<a id="5"></a>
## 5. 연습문제 — 손익분기 크기 찾기

`cp.sort`(GPU)와 `np.sort`(CPU)의 실행시간이 **역전되는 배열 크기**를 찾으세요.
- 여러 크기에 대해 `compare`로 CPU/GPU wall-clock을 측정합니다.
- speedup이 1을 넘어서기 시작하는 N을 보고합니다. (여유가 되면 matplotlib으로 그래프)

이 연습은 2절의 손익분기점 모델을 **정렬**이라는 다른 연산에 적용해보는 것입니다. 원소별 연산은
`O(N)`이지만 정렬은 `O(N log N)`이고, 내부적으로 GPU 정렬(Thrust 기반 radix/merge sort)은 데이터를
여러 차례 오가며 **여러 개의 커널을 순차적으로 실행**합니다 — 즉 2절 모델의 `b0`(고정 오버헤드)가
원소별 연산보다 훨씬 크고, `b1`(원소당 비용)도 `log N` 인자가 섞여 완전한 상수가 아닙니다. 그 결과
정렬의 손익분기 N은 앞서 관찰한 원소별 연산의 손익분기 N과 **다를 가능성이 높습니다** — 더 큰지
작은지는 직접 측정해서 확인하세요.

In [ ]:
sizes = [1_000, 10_000, 100_000, 1_000_000, 10_000_000, 50_000_000]

def find_breakeven(sizes):
    # TODO: 각 크기에서 np.sort vs cp.sort를 compare로 측정하고,
    #       speedup(>1)이 처음 나타나는 N을 반환하세요.
    # breakeven = None
    # ...
    # print('손익분기 크기(대략):', f'{breakeven:,}' if breakeven else '관측 범위 내 없음')
    # return breakeven
    raise NotImplementedError

# find_breakeven(sizes)

<details>
<summary>💡 해답 보기</summary>

```python
def find_breakeven(sizes):
    breakeven = None
    for n in sizes:
        y_np = np.random.rand(n).astype(np.float32)
        y_cp = cp.asarray(y_np)
        c, g, sp = compare(f'N={n:,}',
                           lambda a=y_np: np.sort(a),
                           lambda a=y_cp: cp.sort(a),
                           n_repeat=5, n_warmup=1)
        if breakeven is None and sp >= 1.0:
            breakeven = n
    print('손익분기 크기(대략):', f'{breakeven:,}' if breakeven else '관측 범위 내 없음')
    return breakeven

find_breakeven(sizes)
```

포인트: 작은 N은 런치·전송 오버헤드로 GPU가 느리고, N이 커지면 GPU가 역전합니다.
손익분기 크기는 **연산 종류·GPU 모델**에 따라 달라집니다. 일반적으로 정렬처럼 `O(N log N)`이고
내부적으로 여러 커널을 순차 실행하는 연산은, 2절에서 본 단순 원소별(`O(N)`) 연산보다 고정
오버헤드(`b0`)가 커서 **손익분기 N이 더 크게** 나오는 경향이 있습니다 — 실행 결과를 2절의 표와
비교해보세요. 같은 맥락에서 실험 B(행렬곱, `O(N^3)`)는 반대로 손익분기 N이 훨씬 **작게** 나오는데,
이는 연산량이 데이터 크기보다 훨씬 빠르게 늘어나기 때문입니다.
</details>

## 🧪 추가 연습 & 실험

**연습 A — dtype 처리량**: 같은 연산 `(a*1.1+2).sum()` 을 `float32` vs `float64` GPU 배열로 측정해 시간비를 구하세요.
소비자용 GPU에서 float64는 보통 느립니다.

소비자용(GeForce/RTX 계열) GPU는 FP64(double precision) 연산 유닛을 FP32 대비 **1:32~1:64** 비율로만
갖추고 있어(다이 면적을 게이밍에 중요한 FP32/렌더링에 집중 배분하기 때문), 같은 연산이라도
`float64`가 `float32`보다 수 배~수십 배 느려질 수 있습니다. 반면 데이터센터용 GPU(A100/H100 등)는
FP64 유닛 비율이 **1:2** 수준으로 훨씬 높아 이중정밀도 과학계산(HPC) 워크로드에도 적합하게
설계되어 있습니다. 또한 `float64`는 `float32`보다 **원소당 바이트 수가 2배**라 메모리 대역폭도
그만큼 더 소모하므로, 이 연산이 연산 병목(compute-bound)인지 메모리 병목(memory-bound)인지에 따라
실측 시간비가 2배~수십 배까지 다양하게 나올 수 있습니다. 어떤 GPU에서 실습 중인지에 따라 결과가
크게 달라지는 대표적인 예이니, `print_env()`로 확인한 GPU 모델과 함께 결과를 해석하세요.

In [ ]:
def dtype_ratio(n=20_000_000):
    # TODO: a32(float32), a64(float64)에 같은 연산을 bench 하고 (f64시간/f32시간) 반환
    raise NotImplementedError

# print('f64/f32 시간비:', dtype_ratio())

<details><summary>💡 해답 보기</summary>

```python
def dtype_ratio(n=20_000_000):
    a32 = cp.random.random(n, dtype=cp.float32)
    a64 = cp.random.random(n, dtype=cp.float64)
    r32 = bench(lambda: (a32*1.1+2).sum(), n_repeat=20, name='f32')
    r64 = bench(lambda: (a64*1.1+2).sum(), n_repeat=20, name='f64')
    print_bench(r32); print_bench(r64)
    return gpu_ms(r64) / gpu_ms(r32)
print('f64/f32 시간비:', round(dtype_ratio(), 2))
```

측정된 비율이 정확히 2배(순수 대역폭 차이)보다 크게 나온다면, 이 GPU의 FP64 연산 유닛 비율이
낮아 대역폭이 아니라 **연산 자체가 병목**이라는 신호입니다 — 반대로 2배에 가깝다면 이 연산은
`memory-bound`(대역폭이 병목)이고 FP64 연산 유닛 부족이 아직 드러나지 않은 것입니다. 이 '어느 쪽이
병목인가'라는 질문은 `05_memory_profiling`의 핵심 주제입니다.
</details>

**실험 B — matmul 손익분기**: 정사각 행렬 곱 `A@A` 에서 GPU가 CPU를 이기기 시작하는 N을 찾아보세요.
(예측 먼저: 원소별 연산보다 손익분기 N이 작을까 클까?)

행렬 곱은 원소별 연산과 근본적으로 다른 스케일링을 가집니다: 정사각행렬 `N×N`의 데이터량은
`O(N^2)`이지만 연산량(곱셈+덧셈 횟수)은 `O(N^3)`입니다. 즉 데이터를 한 번 GPU에 올리면(`O(N^2)`
비용) 그 데이터로 **N에 비례해 점점 더 많은 연산**(`O(N^3)`)을 수행하게 되어, 데이터 대비 연산
비중(= **연산 강도, arithmetic intensity**)이 N이 커질수록 함께 커집니다. 연산 강도가 높을수록
GPU의 압도적인 연산 처리량(FLOPS)을 충분히 활용할 수 있어 손익분기점이 **원소별 연산보다 훨씬
작은 N**에서 형성되는 경향이 있습니다 — 게다가 CuPy의 `@`는 NVIDIA가 손수 튜닝한 cuBLAS 커널을
호출하므로 실제 하드웨어 성능에 가깝게 도달합니다. '연산 강도가 병목을 좌우한다'는 이 직관은
**루프라인(roofline) 모델**이라는 이름으로 `05_memory_profiling`에서 정식으로 다룹니다.

In [ ]:
for N in [64, 128, 256, 512, 1024, 2048]:
    A_np = np.random.random((N, N)).astype(np.float32); A_cp = cp.asarray(A_np)
    compare(f'matmul N={N}', lambda A=A_np: A@A, lambda A=A_cp: A@A, n_repeat=5, n_warmup=2)

<a id="6"></a>
## 6. 체크포인트

- [ ] `bench`로 워밍업·동기화·반복평균이 적용된 시간을 측정할 수 있다
- [ ] 작은 배열에서 GPU가 더 느릴 수 있는 이유(런치 오버헤드·전송)를 설명할 수 있다
- [ ] '연산만' vs '연산+전송'의 차이를 수치로 확인했다
- [ ] 파이썬 루프 대신 벡터화를 써야 하는 이유를 안다
- [ ] 연습: `cp.sort`/`np.sort`의 손익분기 크기를 관측했다

다음: **`02_ndarray_core`** — ndarray의 구조·뷰/복사·브로드캐스팅과 NumPy→CuPy 포팅을 깊게 다룹니다.